In [10]:
# https://github.com/arpanghosh8453/programs/blob/master/Fitbit%20Data%20Analyzer/Fitbit%20HR%20analyzer.ipynb
# https://dev.fitbit.com/
# https://dev.fitbit.com/build/reference/web-api/basics/
# Implicit Grant Flow
# You need the following modules, if you don't have them, use pip install <module-name>

import requests
import pandas as pd
import datetime
import numpy as np
import json

import dash
import dash_core_components as dcc
import dash_html_components as html
import plotly.express as px
import jupyter_dash
from dash.dependencies import Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

In [11]:
with open("data/credentials.json", "r") as file:
    credentials = json.load(file)
    fitbit_cr = credentials['fitbit_cr']
    fitbit_key = fitbit_cr['KEY']
    username = fitbit_cr['USERNAME']

In [12]:
# day_range_length = 1

# start_date = str((datetime.datetime.now() - datetime.timedelta(days=day_range_length)).date())
# #start_date = "2020-01-01"

# end_date = str(datetime.datetime.date(datetime.datetime.now())) 
# #end_date = "2020-01-02"

### Date range

In [13]:
start_date = '2020-01-01'
end_date = '2020-01-08'
print(start_date, "to", end_date)

2020-01-01 to 2020-01-08


In [14]:
#Update your start and end dates here in yyyy-mm-dd format 
start = datetime.datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.datetime.strptime(end_date, "%Y-%m-%d")

date_array = (start + datetime.timedelta(days=x) for x in range(0, (end-start).days))

day_list = []
for date_object in date_array:
    day_list.append(date_object.strftime("%Y-%m-%d"))
    
print("day range : ", day_list)

day range :  ['2020-01-01', '2020-01-02', '2020-01-03', '2020-01-04', '2020-01-05', '2020-01-06', '2020-01-07']


In [16]:
df_all = pd.DataFrame()

### Heart Rate requests

In [18]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

for single_day in day_list:
    response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+ single_day +"/1d/1min/time/00:00/23:59.json", headers=header).json()
    try:
        df = pd.DataFrame(response['activities-heart-intraday']['dataset'])
        date = pd.Timestamp(single_day).strftime('%Y-%m-%d')
        df = df.set_index(pd.to_datetime(date + ' ' + df['time'].astype(str)))
        #print(df)
        df_all = df_all.append(df, sort=True)
    except KeyError as e:
        print("No data for the given date", date)
    
# df_all.index.set_names('dateTime', inplace = True)   
# del df_all['time']
df_all

In [20]:
#del df_all['time']

In [21]:
#df_all

,value
time,
2020-01-01 00:00:00,57
2020-01-01 00:01:00,57
2020-01-01 00:02:00,56
2020-01-01 00:03:00,56
2020-01-01 00:04:00,57
...,...
2020-01-07 23:55:00,72
2020-01-07 23:56:00,76
2020-01-07 23:57:00,75


In [13]:
# #Put the interval you want to take the average of the imported data from fitbit with 2-5 sec interval. Default 10 minute
# summary_df = (df_all['value'].resample('10Min').mean())
# summary_df

In [55]:
#Put the interval you want to take the average of the imported data from fitbit with 2-5 sec interval. Default 10 minute
summary_df = df_all
summary_df

,value,time
time,,
2020-01-01 00:00:00,57,2020-01-01 00:00:00
2020-01-01 00:01:00,57,2020-01-01 00:01:00
2020-01-01 00:02:00,56,2020-01-01 00:02:00
2020-01-01 00:03:00,56,2020-01-01 00:03:00
2020-01-01 00:04:00,57,2020-01-01 00:04:00
...,...,...
2020-01-07 23:55:00,72,2020-01-07 23:55:00
2020-01-07 23:56:00,76,2020-01-07 23:56:00
2020-01-07 23:57:00,75,2020-01-07 23:57:00


In [335]:
# # trying to create empty <None> list so that gaps show up in the heart rate dashboard when I don't have it on
# blank_range = pd.DataFrame()
# blank_range['time'] = pd.date_range("1/1/2020", "1/8/2020", freq="T")
# #blank_range['time'].to_string()
# blank_range['value'] = None
# blank_range

In [323]:
df2 = pd.read_csv('~/qs/fitbit/data/hr.csv')
df2

,value,time
0,57,2020-01-01 00:00:00
1,57,2020-01-01 00:01:00
2,56,2020-01-01 00:02:00
3,56,2020-01-01 00:03:00
4,57,2020-01-01 00:04:00
...,...,...
9658,72,2020-01-07 23:55:00
9659,76,2020-01-07 23:56:00
9660,75,2020-01-07 23:57:00
9661,76,2020-01-07 23:58:00


In [324]:
type(df2['time'][0])

str

In [325]:
df2['time'] = pd.to_datetime(df2['time'])
type(df2['time'][0])

pandas._libs.tslibs.timestamps.Timestamp

In [327]:
df2 = df2.set_index('time')
df2

,value
time,
2020-01-01 00:00:00,57
2020-01-01 00:01:00,57
2020-01-01 00:02:00,56
2020-01-01 00:03:00,56
2020-01-01 00:04:00,57
...,...
2020-01-07 23:55:00,72
2020-01-07 23:56:00,76
2020-01-07 23:57:00,75


#### How to set the missing timestamp values as None

In [329]:
# https://stackoverflow.com/questions/51656065/pandas-resampling-typeerror-only-valid-with-datetimeindex-timedeltaindex-or-p
# https://stackoverflow.com/questions/54592536/find-gaps-in-pandas-time-series-dataframe-sampled-at-1-minute-intervals-and-fill
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.resample.html
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.asfreq.html#pandas.DataFrame.asfreq
df2 = df2.resample('T', convention='start').asfreq()
df2

,value
time,
2020-01-01 00:00:00,57.0
2020-01-01 00:01:00,57.0
2020-01-01 00:02:00,56.0
2020-01-01 00:03:00,56.0
2020-01-01 00:04:00,57.0
...,...
2020-01-07 23:55:00,72.0
2020-01-07 23:56:00,76.0
2020-01-07 23:57:00,75.0


In [333]:
df2['time'] = df2.index
df2

,value,time
time,,
2020-01-01 00:00:00,57.0,2020-01-01 00:00:00
2020-01-01 00:01:00,57.0,2020-01-01 00:01:00
2020-01-01 00:02:00,56.0,2020-01-01 00:02:00
2020-01-01 00:03:00,56.0,2020-01-01 00:03:00
2020-01-01 00:04:00,57.0,2020-01-01 00:04:00
...,...,...
2020-01-07 23:55:00,72.0,2020-01-07 23:55:00
2020-01-07 23:56:00,76.0,2020-01-07 23:56:00
2020-01-07 23:57:00,75.0,2020-01-07 23:57:00


In [331]:
# from IPython.display import display
# with pd.option_context('display.max_rows', 10500):
#     display(df2)

In [22]:
# import matplotlib.pyplot as plt

# plt.rcParams["figure.figsize"]=20,8

# # Heart rate data summary [10min avg] from start date[2021-03-18] to end date[2021-03-21] 
# #if you are using matplotlib directly in python ( py file ) then use plt.plot(summary_df,kind='area')
# summary_df.plot.area(ylim=(40,160))

In [334]:
df2.to_csv('data/hr2.csv', index=None, encoding='utf-8')

### Resting Heart Rate requests

In [17]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+start_date+"/"+end_date+".json", headers=header).json()

In [24]:
#response

In [25]:
#all_resting_HR_list = []
for i in response['activities-heart']:
    try:
        resting_dict = { 'dateTime':i['dateTime'], "resting_HR":i['value']['restingHeartRate']}
        all_resting_HR_list.append(resting_dict)
    except KeyError as e:
        print("No data for the given date", i['dateTime'])
    
resting_HR_df = pd.DataFrame(all_resting_HR_list)
resting_HR_df['time'] = resting_HR_df.dateTime.apply(pd.Timestamp)
 
# resting_HR_df.set_index("dateTime", inplace = True)
# resting_HR_df['time'] = resting_HR_df.index

In [27]:
del resting_HR_df['dateTime']
resting_HR_df

,resting_HR,time
0,56,2020-01-01
1,58,2020-01-02
2,57,2020-01-03
3,58,2020-01-04
4,57,2020-01-05
5,58,2020-01-06
6,59,2020-01-07
7,58,2020-01-08
8,56,2020-01-01
9,58,2020-01-02


### Step Count

In [28]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1/user/-/activities/steps/date/"+start_date+"/"+end_date+"/1min.json", headers=header).json()['activities-steps']

In [29]:
step_df = pd.DataFrame(response)
step_df['time'] = pd.to_datetime(step_df['dateTime'].apply(pd.Timestamp)).dt.date
del step_df['dateTime']
step_df['Step Count'] = step_df['value'].apply(int)
del step_df['value']
step_df

,dateTime,time,Step Count
0,2020-01-01,2020-01-01,1041
1,2020-01-02,2020-01-02,321
2,2020-01-03,2020-01-03,1659
3,2020-01-04,2020-01-04,8407
4,2020-01-05,2020-01-05,4520
5,2020-01-06,2020-01-06,3202
6,2020-01-07,2020-01-07,1542
7,2020-01-08,2020-01-08,4072


### Distance

In [39]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1/user/-/activities/distance/date/"+start_date+"/"+end_date+"/1min.json", headers=header).json()['activities-distance']

In [40]:
distance_df = pd.DataFrame(response)
distance_df['time'] = pd.to_datetime(distance_df['dateTime'].apply(pd.Timestamp)).dt.date
del distance_df['dateTime']
distance_df['Distance in KM'] = distance_df['value'].apply(float)
del distance_df['value']
distance_df

,time,Distance in KM
0,2020-01-01,0.82576
1,2020-01-02,0.24246
2,2020-01-03,1.25603
3,2020-01-04,6.37055
4,2020-01-05,3.49040
5,2020-01-06,2.31321
6,2020-01-07,1.16396
7,2020-01-08,3.08456


### Sleep

In [336]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1.2/user/-/sleep/date/"+start_date+"/"+end_date+".json", headers=header).json()

In [338]:
response

{'sleep': [{'dateOfSleep': '2020-01-08',
   'duration': 38460000,
   'efficiency': 98,
   'endTime': '2020-01-08T16:04:00.000',
   'infoCode': 0,
   'isMainSleep': True,
   'levels': {'data': [{'dateTime': '2020-01-08T05:23:00.000',
      'level': 'wake',
      'seconds': 2820},
     {'dateTime': '2020-01-08T06:10:00.000', 'level': 'light', 'seconds': 150},
     {'dateTime': '2020-01-08T06:12:30.000', 'level': 'wake', 'seconds': 300},
     {'dateTime': '2020-01-08T06:17:30.000', 'level': 'light', 'seconds': 840},
     {'dateTime': '2020-01-08T06:31:30.000', 'level': 'wake', 'seconds': 780},
     {'dateTime': '2020-01-08T06:44:30.000', 'level': 'light', 'seconds': 840},
     {'dateTime': '2020-01-08T06:58:30.000', 'level': 'deep', 'seconds': 1050},
     {'dateTime': '2020-01-08T07:16:00.000', 'level': 'light', 'seconds': 390},
     {'dateTime': '2020-01-08T07:22:30.000', 'level': 'rem', 'seconds': 1350},
     {'dateTime': '2020-01-08T07:45:00.000',
      'level': 'light',
      'seconds

In [350]:
sleep_duration_list = []
for i in response['sleep']:
    sleep_duration_list.append({'date':i['dateOfSleep'], 'start_time':i['startTime'], 'end_time':i['endTime']})
    
sleep_df = pd.DataFrame(sleep_duration_list)

In [351]:
sleep_df

,date,start_time,end_time
0,2020-01-08,2020-01-08T05:23:00.000,2020-01-08T16:04:00.000
1,2020-01-07,2020-01-06T19:44:00.000,2020-01-07T07:06:30.000
2,2020-01-06,2020-01-06T01:35:00.000,2020-01-06T08:55:30.000
3,2020-01-05,2020-01-05T10:00:30.000,2020-01-05T14:25:00.000
4,2020-01-04,2020-01-04T09:40:30.000,2020-01-04T13:49:30.000
5,2020-01-03,2020-01-03T12:55:00.000,2020-01-03T21:03:30.000
6,2020-01-03,2020-01-02T15:47:00.000,2020-01-03T03:55:30.000
7,2020-01-02,2020-01-01T20:40:00.000,2020-01-02T07:12:30.000
8,2020-01-01,2019-12-31T23:00:00.000,2020-01-01T03:58:00.000


In [353]:
sleep_df['start_time'] = sleep_df['start_time'].to_timestamp()
sleep_df

TypeError: unsupported Type RangeIndex

In [354]:
type(sleep_df['start_time'][0])

str

In [358]:
sleep_df['start_time'] = sleep_df['start_time'].str.strip('T', ' ')
sleep_df

TypeError: strip() takes from 1 to 2 positional arguments but 3 were given